# Multi-line CSV

CSV files sometimes contain fields with embedded newlines. If these fields aren't properly quoted, pandas will interpret the newline as a row boundary, corrupting the data. Columns shift, data types break, and downstream processing fails silently or with confusing errors.

In this notebook, we will learn to:
1. **Detect** corrupted rows caused by unquoted newlines in CSV fields
2. **Understand** why pandas misinterprets these lines
3. **Fix** the problem by pre-processing the file before loading it into a DataFrame

## Exercise 1 - Load the CSV

Load `BusinessRules.csv` using `pd.read_csv()`. Print the column names and the shape of the DataFrame.

This gives us a first look at the data and tells us how many rows and columns pandas found.

**Explanation:** We use `pd.read_csv()` to load the CSV file into a DataFrame. The `.columns` attribute returns the column names, and `.shape` returns a tuple of `(rows, columns)`. This is a standard first step when exploring any new dataset.

In [ ]:
import pandas as pd

df = pd.read_csv('BusinessRules.csv')

print(df.columns)
print(df.shape)

## Exercise 2 - A properly quoted newline

Row 1377 (by index) in the `EnrolleeContractRateDeterminationRule` column contains an embedded newline (`\n`). However, this field is properly quoted in the CSV file, so pandas handles it correctly.

Display rows 1375 through 1380 of the `EnrolleeContractRateDeterminationRule` column using `.loc` and `.values` to see the raw content. You should be able to spot the `\n` character in row 1377.

**Explanation:** We use `.loc` with a list of index values to select specific rows, and pass a single column name in a list to keep it as a DataFrame slice. Calling `.values` returns a NumPy array, which displays the raw string content including escape characters like `\n`. This makes it easy to spot the embedded newline in row 1377 -- note that the surrounding rows do not have it.

In [ ]:
df.loc[[1375, 1376, 1377, 1378, 1379, 1380], ["EnrolleeContractRateDeterminationRule"]].values

## Exercise 3 - Spot the corrupted row

Now inspect rows 9260 through 9262 by displaying **all columns** using `.loc`. Look carefully at row 9261 -- the `BusinessYear` column contains text that clearly does not belong there. The data from one row has "spilled" into the next because of an unquoted newline in the CSV.

Display these rows so you can see the problem.

**Explanation:** We use `.loc` with a list of indices to display three consecutive rows. Row 9261 clearly shows the corruption: its `BusinessYear` column contains the text `"for each enrollee is added together"` instead of a year. This happened because the CSV had an unquoted newline inside a field of the previous row, causing pandas to treat the continuation as a brand new row. All column values in row 9261 are shifted as a result.

In [ ]:
df.loc[[9260, 9261, 9262]]

## Exercise 4 - Confirm the corruption with a type conversion

The `BusinessYear` column should contain only integer year values (like `2015`). Try to convert it to integer using `.astype(int)`. This will raise a `ValueError` because row 9261 contains text instead of a number.

Wrap the conversion in a `try/except ValueError` block and print the error message.

**Explanation:** Attempting `.astype(int)` on a column that should be purely numeric is a quick way to surface data corruption. pandas will raise a `ValueError` when it encounters a value that cannot be converted to an integer. The error message helpfully includes the offending value, making it easy to trace back to the problematic row. Using `try/except` prevents the notebook from stopping on the error.

In [ ]:
try:
    df["BusinessYear"].astype(int)
except ValueError as e:
    print(f"Error: {e}")

## Pre-processing approach

We can fix this by pre-processing the CSV file before passing it to pandas. The idea is:

1. Read the file **line by line**
2. For each line, check whether the **first comma-separated field** is numeric (since `BusinessYear` should always be a year like `2015`)
3. If it is numeric, it is a proper new row -- write it to the output
4. If it is **not** numeric, the line is a continuation of the previous row -- **append** it to the previous line
5. Use `io.StringIO` to collect the cleaned lines into an in-memory file object
6. Pass the `StringIO` object to `pd.read_csv()`

This approach avoids modifying the original file and keeps everything in memory.

```python
import io

# StringIO creates an in-memory text stream that behaves like a file
buffer = io.StringIO()
buffer.write("some text\n")
buffer.seek(0)  # rewind to the beginning before reading
print(buffer.read())
```

**Note:** This method is not bulletproof -- if another field's newline happens to leave a valid numeric value in the first column, it would go undetected. But it handles the majority of cases and gives you a solid foundation to build on.

## Exercise 5 - Write the cleaning function

Write a function `read_and_clean_csv(file_path)` that:

1. Creates an `io.StringIO` object to hold the cleaned CSV
2. Opens the file and reads the first line into a variable `prev_line` (this is the header)
3. Iterates over the remaining lines:
   - If the first comma-separated field (`line.split(',')[0]`) is numeric (`.isnumeric()`), write `prev_line` to the buffer and set `prev_line` to the current line
   - Otherwise, append the current line to `prev_line` (strip both and join with a space)
4. After the loop, writes the final `prev_line` to the buffer
5. Seeks back to position 0
6. Returns `pd.read_csv()` of the `StringIO` object

**Explanation:** The function reads the file line by line and uses a simple heuristic: if the first comma-separated field is numeric, the line is a legitimate new row; otherwise, it is a continuation of the previous row caused by an unquoted newline. We accumulate the cleaned lines in an `io.StringIO` buffer, which behaves like a file object and can be passed directly to `pd.read_csv()`. The header line is read separately with `f.readline()` because it is not numeric and would otherwise be merged into the last data row. After the loop, we must remember to write the final `prev_line` since it is only written when the next valid line arrives.

In [ ]:
import io

def read_and_clean_csv(file_path):
    cleaned_csv = io.StringIO()

    with open(file_path, 'r', encoding='utf-8') as f:
        prev_line = f.readline()
        for line in f:
            if not line.split(',')[0].isnumeric():
                prev_line = prev_line.strip() + ' ' + line.strip()
            else:
                cleaned_csv.write(prev_line + '\n')
                prev_line = line

    cleaned_csv.write(prev_line + '\n')
    cleaned_csv.seek(0)
    return pd.read_csv(cleaned_csv)

## Exercise 6 - Verify the fix

Load the CSV using your `read_and_clean_csv()` function. Then:

1. Print the shape of the new DataFrame
2. Convert the `BusinessYear` column to `int` -- this should now succeed without errors
3. Display rows 9260 through 9262 to confirm the data is no longer corrupted

**Explanation:** After loading the CSV with our cleaning function, the corrupted row 9261 has been merged back into the previous row where it belongs. The shape should show one fewer row than before, because the spurious row has been eliminated. The `.astype(int)` conversion now succeeds because every value in `BusinessYear` is a proper year. Displaying rows 9260-9262 confirms that all columns contain sensible values again.

In [ ]:
df_clean = read_and_clean_csv('BusinessRules.csv')

print(df_clean.shape)

# This should now work without errors
try:
    df_clean["BusinessYear"].astype(int)
    print("BusinessYear successfully converted to int!")
except ValueError as e:
    print(f"Error: {e}")

# Check the previously problematic rows
df_clean.loc[[9260, 9261, 9262]]

## Summary

In this notebook we learned:

- CSV fields with embedded newlines can corrupt a DataFrame when the field is not properly quoted
- pandas may silently load the data but shift column values, making the corruption hard to spot until you try to use the data
- A type conversion (e.g. `.astype(int)`) on a column that should be numeric is a quick way to detect this kind of corruption
- Pre-processing the file line by line with `io.StringIO` lets you merge broken lines before pandas ever sees them
- The key heuristic is checking whether the first field of each line matches the expected pattern (numeric in this case) to decide if a line is a real new row or a continuation

For further reading, see the [pandas `read_csv` documentation](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) and the [Python `io` module documentation](https://docs.python.org/3/library/io.html).